<font size="6" color='grey'> <b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

---

<font size="5" color='grey'> <b>
M04a - Übung A1: Simple Chain mit Parser
</b></font> </br>

**Lernziel:** Verstehe wie man eine einfache LangChain-Kette mit Prompt, LLM und Parser aufbaut und anwendet.

## Setup & Environment

Umgebung vorbereiten und erforderliche Module laden.

In [ ]:
#@title 🔧 Umgebung einrichten (LOCAL VERSION)
# LOKAL: genai_lib muss bereits installiert sein
# Falls nicht: pip install -e /Users/wagnerg/Development/playground/GenAI_GW/lessons/GenAI/04_modul

import subprocess
import sys

# python-dotenv sicherstellen
try:
    from dotenv import load_dotenv
except ImportError:
    print("📦 Installiere python-dotenv...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv

import os

# API Keys aus .env laden
env_path = '/Users/wagnerg/Development/playground/GenAI_GW/.env'
load_dotenv(env_path)

# Imports
from genai_lib.utilities import check_environment, mprint

print("✅ Umgebung wird vorbereitet...")
print()
check_environment()
print()
print(f"✓ OPENAI_API_KEY gesetzt: {'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY'] != ''}")
print(f"✓ genai_lib importiert erfolgreich")

## Imports

Erforderliche LangChain-Komponenten importieren.

In [ ]:
# Importe für Simple Chain mit Parser
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers.string import StrOutputParser

## Model-Konfiguration

Parameter und Modell-Initialisierung.

In [ ]:
# Parameter
model_provider = "openai"
model_name = "gpt-4o-mini"
temperature = 0.0

# Modell definieren
llm = init_chat_model(model_name, model_provider=model_provider, temperature=temperature)

print(f"✅ Modell initialisiert: {model_name}")
print(f"   Temperature: {temperature}")

# Übung: Simple Chain mit Parser

Eine einfache Verarbeitungskette aufbauen:
- **Prompt**: Definiert die Eingabe-Struktur
- **LLM**: Verarbeitet die Anfrage
- **Parser**: Konvertiert die Ausgabe zu einem String

```
Prompt → LLM → Parser
  |       |       |
 Input  Process  Output
```

## Beispiel 1: Chain für Erklärungen

In [ ]:
# 1. Prompt-Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist ein hilfreicher und humorvoller Assistent."),
    ("human", "Erkläre mir {user_input}")
])

print("✅ ChatPromptTemplate erstellt")

In [ ]:
# 3. Parser
parser = StrOutputParser()

print("✅ StrOutputParser erstellt")

In [ ]:
# 4. Einfache LCEL-Kette mit Pipe-Operator
chain = prompt | llm | parser

print("✅ LCEL-Kette erstellt: Prompt → LLM → Parser")

In [ ]:
# 5. Ausführung
response = chain.invoke({"user_input": "LangChain Expression Language"})

mprint("## 📣 Model response:")
mprint("---")
mprint(response)

## Beispiel 2: Chain mit verschiedenen Topics

In [ ]:
# Verschiedene Anfragen testen
topics = [
    "Generative KI",
    "Prompt Engineering",
    "Embeddings"
]

for topic in topics:
    print(f"\n" + "="*60)
    print(f"Topic: {topic}")
    print("="*60)
    
    response = chain.invoke({"user_input": topic})
    # Nur die ersten 300 Zeichen anzeigen für Überblick
    preview = response[:300] + "..." if len(response) > 300 else response
    print(preview)

## Beispiel 3: Chain mit angepasstem System-Prompt

In [ ]:
# Alternative Chain mit technischerem Tonfall
technical_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Benutze so viele Emojis wie möglich.
Du bist ein technischer Experte. 
Antworte präzise und fachlich korrekt.
Verwende technische Begriffe.
Halte die Antwort auf 2-3 Sätze begrenzt."""),
    ("human", "Erkläre: {user_input}")
])

# Chain zusammenstellen
technical_chain = technical_prompt | llm | parser

print("✅ Technical Chain erstellt")

In [ ]:
# Test: Vergleich Standard vs. Technical
test_topic = "Token-Limit bei LLMs"

print("\n" + "="*60)
print(f"Topic: {test_topic}")
print("="*60)

# Standard Chain
print("\n🎨 Standard Chain (humorvoller Assistent):")
print("-" * 40)
response_standard = chain.invoke({"user_input": test_topic})
print(response_standard[:200] + "...")

# Technical Chain
print("\n🔧 Technical Chain (technischer Experte):")
print("-" * 40)
response_technical = technical_chain.invoke({"user_input": test_topic})
print(response_technical)

## Beispiel 4: Chain mit Batch-Verarbeitung

In [ ]:
# Batch-Verarbeitung: Mehrere Anfragen auf einmal
batch_inputs = [
    {"user_input": "RAG (Retrieval Augmented Generation)"},
    {"user_input": "Fine-Tuning von Modellen"},
    {"user_input": "Chain-of-Thought Prompting"}
]

print("📦 Batch-Verarbeitung starten...\n")

# batch() macht mehrere Anfragen parallel
responses = chain.batch(batch_inputs)

for i, (inp, resp) in enumerate(zip(batch_inputs, responses), 1):
    print(f"\n{i}. {inp['user_input']}")
    print("-" * 50)
    preview = resp[:150] + "..." if len(resp) > 150 else resp
    print(preview)

## Beispiel 5: Chain mit Stream-Verarbeitung

In [ ]:
# Stream-Verarbeitung: Echtzeitausgabe
print("🌊 Stream-Verarbeitung (Token-by-Token):")
print("-" * 50)

stream_input = {"user_input": "Nenne die 3 wichtigsten Konzepte in LangChain"}

# stream() gibt Token-für-Token zurück, sobald verfügbar
for chunk in chain.stream(stream_input):
    print(chunk, end="", flush=True)  # Echtzeit-Ausgabe

print("\n")

## 💡 Erkenntnisse

### Was passiert in einer LCEL-Chain?

```python
chain = prompt | llm | parser
```

1. **Prompt** (`ChatPromptTemplate`)
   - Nimmt die Eingabe `{"user_input": ...}` entgegen
   - Formatiert sie nach dem Template-Schema (system + human)
   - Gibt Nachrichten-Liste aus

2. **LLM** (`init_chat_model`)
   - Empfängt die Nachrichten-Liste vom Prompt
   - Sendet Anfrage an OpenAI API
   - Gibt `AIMessage`-Objekt zurück

3. **Parser** (`StrOutputParser`)
   - Nimmt das `AIMessage`-Objekt entgegen
   - Extrahiert nur den Text-Content
   - Gibt einen **reinen String** zurück

### Unterschied zwischen Runnables

| Methode | Verwendung | Rückgabe |
|---------|-----------|----------|
| `.invoke()` | Eine Anfrage synchron | String (direkt) |
| `.batch()` | Mehrere Anfragen parallel | Liste von Strings |
| `.stream()` | Echtzeit-Token-Streaming | Generator von Text-Chunks |

### System-Prompt vs. User-Prompt

- **System-Prompt**: Definiert Verhalten & Rolle ("Du bist ein hilfreicher Assistent")
- **User-Prompt**: Konkrete Aufgabe ("Erkläre mir {user_input}")

## 📝 Zusammenfassung

✅ **Was wir gelernt haben:**

1. **LCEL-Syntax**: Mit dem Pipe-Operator `|` komponieren wir Komponenten
2. **ChatPromptTemplate**: Strukturierte Prompts mit Rollen und Variablen
3. **StrOutputParser**: Konvertiert AIMessage zu reinem String
4. **Runnable-Methoden**: `.invoke()` für einzelne, `.batch()` für mehrere, `.stream()` für Echtzeit
5. **Systemverhalten anpassen**: Über den System-Prompt steuern wir Tonfall und Expertise

🚀 **Nächste Schritte:**
- Erkunde andere Parser (z.B. `JsonOutputParser`)
- Experimentiere mit verschiedenen Prompts
- Kombiniere mehrere Chains hintereinander